# Modelling

Owner: 

Input: 

Output: 

In [6]:
import sys
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

# Identify the project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Root exists:", ROOT.exists())

Project root: d:\Gamage\marketing mix modeling
Root exists: True


In [7]:
CONFIG_PATH = ROOT / "config" / "params.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration file not found: {CONFIG_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = json.load(file)

DATE_COLUMN = config["date_col"]
TARGET = config["target"]
CHANNELS = config["channels"]
EXTERNAL_FACTORS = config["external_factors"]
TEST_SIZE = config["test_size_months"]

print("Target:", TARGET)
print("Media channels:", len(CHANNELS))
print("External factors:", len(EXTERNAL_FACTORS))
print("Test months:", TEST_SIZE)

Target: sales_revenue_lkr
Media channels: 6
External factors: 8
Test months: 6


In [9]:
%pip install openpyxl


   ---------------------------------------- 0/2 [et-xmlfile]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [open


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
DATA_PATH = ROOT / config["data_file"]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

df = pd.read_excel(
    DATA_PATH,
    sheet_name=config["sheet"],
    engine="openpyxl"
)

df[DATE_COLUMN] = pd.to_datetime(
    df[DATE_COLUMN],
    errors="raise"
)

df = (
    df.sort_values(DATE_COLUMN)
      .reset_index(drop=True)
)

print("Dataset shape:", df.shape)
print(
    "Date range:",
    df[DATE_COLUMN].min(),
    "to",
    df[DATE_COLUMN].max()
)

display(df.head())

Dataset shape: (36, 25)
Date range: 2023-01-01 00:00:00 to 2025-12-01 00:00:00


,month,country,brand_category,region_scope,tv_spend_lkr,facebook_instagram_spend_lkr,google_search_spend_lkr,youtube_spend_lkr,radio_spend_lkr,outdoor_spend_lkr,...,conversions,average_price_index,promotion_discount_pct,competitor_pressure_index_0_100,economic_sentiment_index_0_100,lkr_usd_exchange_rate,inflation_proxy_index,seasonal_demand_index,public_holiday_count,data_type
0,2023-01-01,Sri Lanka,FMCG - Packaged Foods,National,4274762,3890003,2467793,1836858,1700676,1813444,...,1210051,0.998,7.27,49.6,56.8,324.42,1.000,1.08,2,SYNTHETIC
1,2023-02-01,Sri Lanka,FMCG - Packaged Foods,National,4023574,5062635,2962576,2254198,1922384,2941194,...,664746,1.017,9.47,51.0,58.4,335.92,1.012,0.96,6,SYNTHETIC
2,2023-03-01,Sri Lanka,FMCG - Packaged Foods,National,5359091,3746337,4080535,2363737,1720670,2063917,...,380625,1.011,6.83,45.9,60.6,328.66,1.024,1.00,2,SYNTHETIC
3,2023-04-01,Sri Lanka,FMCG - Packaged Foods,National,5102550,3690115,3390558,2595121,1217450,3091430,...,1228413,1.017,21.52,31.2,54.5,325.60,1.037,1.19,6,SYNTHETIC
4,2023-05-01,Sri Lanka,FMCG - Packaged Foods,National,4972723,5333813,3240078,2343364,1568285,3525300,...,562986,1.014,1.95,53.0,56.5,330.03,1.049,1.10,3,SYNTHETIC


In [11]:
required_columns = (
    [DATE_COLUMN, TARGET]
    + CHANNELS
    + EXTERNAL_FACTORS
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if df[DATE_COLUMN].duplicated().any():
    raise ValueError("Duplicate monthly dates found.")

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values found in required columns. "
        "Inspect and handle them before modelling."
    )

if not np.isfinite(
    df[required_columns]
      .drop(columns=[DATE_COLUMN])
      .to_numpy(dtype=float)
).all():
    raise ValueError("Infinite or invalid numeric values found.")

if (df[CHANNELS] < 0).any().any():
    raise ValueError("Negative media spend found.")

print("Data validation passed.")
print("Rows:", len(df))
print("Unique months:", df[DATE_COLUMN].nunique())

Data validation passed.
Rows: 36
Unique months: 36


In [12]:
if len(df) <= TEST_SIZE:
    raise ValueError("Not enough rows for the requested test period.")

split_index = len(df) - TEST_SIZE

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

y_train = train_df[TARGET].astype(float)
y_test = test_df[TARGET].astype(float)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print(
    "Training period:",
    train_df[DATE_COLUMN].min().date(),
    "to",
    train_df[DATE_COLUMN].max().date()
)

print(
    "Testing period:",
    test_df[DATE_COLUMN].min().date(),
    "to",
    test_df[DATE_COLUMN].max().date()
)

assert len(test_df) == 6
assert train_df[DATE_COLUMN].max() < test_df[DATE_COLUMN].min()

Training rows: 30
Testing rows: 6
Training period: 2023-01-01 to 2025-06-01
Testing period: 2025-07-01 to 2025-12-01


In [13]:
def apply_adstock(series, decay):
    values = series.astype(float).to_numpy()
    result = np.zeros(len(values), dtype=float)

    for i in range(len(values)):
        previous = result[i - 1] if i > 0 else 0.0
        result[i] = values[i] + decay * previous

    return pd.Series(result, index=series.index)


adstock_df = df[[DATE_COLUMN, TARGET]].copy()

for channel in CHANNELS:
    decay = config["adstock_decay"][channel]

    adstock_df[f"{channel}_adstock"] = apply_adstock(
        df[channel],
        decay
    )

display(adstock_df.head())

,month,sales_revenue_lkr,tv_spend_lkr_adstock,facebook_instagram_spend_lkr_adstock,google_search_spend_lkr_adstock,youtube_spend_lkr_adstock,radio_spend_lkr_adstock,outdoor_spend_lkr_adstock
0,2023-01-01,68727157,4.274762e+06,3.890003e+06,2.467793e+06,1.836858e+06,1.700676e+06,1.813444e+06
1,2023-02-01,73594547,5.733479e+06,5.840636e+06,3.209355e+06,2.805255e+06,2.262519e+06,3.485227e+06
2,2023-03-01,85063287,7.652483e+06,4.914464e+06,4.401471e+06,3.205314e+06,2.173174e+06,3.109485e+06
3,2023-04-01,100712194,8.163543e+06,4.673008e+06,3.830705e+06,3.556715e+06,1.652085e+06,4.024276e+06
4,2023-05-01,77502288,8.238140e+06,6.268415e+06,3.623149e+06,3.410379e+06,1.898702e+06,4.732583e+06


In [14]:
def hill_saturation(series, half_sat, slope):
    x = np.maximum(
        series.astype(float).to_numpy(),
        0.0
    )

    if half_sat <= 0:
        raise ValueError("half_sat must be positive.")

    if slope <= 0:
        raise ValueError("slope must be positive.")

    numerator = np.power(x, slope)
    denominator = numerator + np.power(half_sat, slope)

    result = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=float),
        where=denominator != 0
    )

    return pd.Series(result, index=series.index)


TRANSFORM_PARAMS = {}
feature_df = df[[DATE_COLUMN, TARGET]].copy()

for channel in CHANNELS:
    adstock_column = f"{channel}_adstock"
    saturation_config = config["saturation"][channel]

    adstock_values = adstock_df[adstock_column]

    # Estimate half-saturation from training data only
    half_sat = saturation_config["half_sat"]

    if half_sat is None:
        half_sat = float(
            adstock_values.iloc[:split_index].median()
        )

    slope = float(saturation_config["slope"])

    feature_name = f"{channel}_hill"

    feature_df[feature_name] = hill_saturation(
        adstock_values,
        half_sat,
        slope
    )

    TRANSFORM_PARAMS[channel] = {
        "adstock_decay": float(
            config["adstock_decay"][channel]
        ),
        "half_sat": float(half_sat),
        "slope": slope
    }

print("Transformation parameters:")
display(pd.DataFrame(TRANSFORM_PARAMS).T)

Transformation parameters:


,adstock_decay,half_sat,slope
tv_spend_lkr,0.4,7.186472e+06,1.0
facebook_instagram_spend_lkr,0.2,5.468062e+06,1.0
google_search_spend_lkr,0.1,3.554542e+06,1.0
youtube_spend_lkr,0.3,3.064299e+06,1.0
radio_spend_lkr,0.2,1.949689e+06,1.0
outdoor_spend_lkr,0.3,3.565590e+06,1.0


In [15]:
MEDIA_FEATURES = [
    f"{channel}_hill"
    for channel in CHANNELS
]

FEATURES = MEDIA_FEATURES + EXTERNAL_FACTORS

X = pd.concat(
    [
        feature_df[MEDIA_FEATURES],
        df[EXTERNAL_FACTORS].astype(float)
    ],
    axis=1
)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

print("Number of predictors:", len(FEATURES))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

display(X_train.head())

Number of predictors: 14
X_train shape: (30, 14)
X_test shape: (6, 14)


,tv_spend_lkr_hill,facebook_instagram_spend_lkr_hill,google_search_spend_lkr_hill,youtube_spend_lkr_hill,radio_spend_lkr_hill,outdoor_spend_lkr_hill,average_price_index,promotion_discount_pct,competitor_pressure_index_0_100,economic_sentiment_index_0_100,lkr_usd_exchange_rate,inflation_proxy_index,seasonal_demand_index,public_holiday_count
0,0.372976,0.415685,0.409773,0.374780,0.465892,0.337132,0.998,7.27,49.6,56.8,324.42,1.000,1.08,2.0
1,0.443769,0.516473,0.474483,0.477933,0.537134,0.494301,1.017,9.47,51.0,58.4,335.92,1.012,0.96,6.0
2,0.515702,0.473340,0.553226,0.511246,0.527103,0.465835,1.011,6.83,45.9,60.6,328.66,1.024,1.00,2.0
3,0.531826,0.460800,0.518697,0.537186,0.458686,0.530217,1.017,21.52,31.2,54.5,325.60,1.037,1.19,6.0
4,0.534091,0.534097,0.504779,0.526726,0.493376,0.570316,1.014,1.95,53.0,56.5,330.03,1.049,1.10,3.0


In [16]:
# Baseline: predict the training mean
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)


# OLS regression
ols_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regression", LinearRegression())
])

ols_model.fit(X_train, y_train)
ols_pred = ols_model.predict(X_test)


# Ridge regression with time-series cross-validation
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regression", Ridge())
])

tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    "regression__alpha": [
        0.01, 0.1, 1.0, 10.0, 100.0
    ]
}

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    refit=True
)

ridge_search.fit(X_train, y_train)

best_model = ridge_search.best_estimator_
ridge_pred = best_model.predict(X_test)

print("Best Ridge alpha:", ridge_search.best_params_)
print("Best CV RMSE:", np.sqrt(-ridge_search.best_score_))

Best Ridge alpha: {'regression__alpha': 10.0}
Best CV RMSE: 4382371.93181566


In [17]:
def calculate_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted),
        "MAPE_pct": np.mean(
            np.abs(
                (np.asarray(actual) - np.asarray(predicted))
                / np.maximum(np.abs(np.asarray(actual)), 1e-8)
            )
        ) * 100
    }


metrics = []

for model_name, predictions in [
    ("Mean baseline", baseline_pred),
    ("OLS regression", ols_pred),
    ("Ridge regression", ridge_pred),
]:
    row = {
        "Model": model_name,
        **calculate_metrics(y_test, predictions)
    }
    metrics.append(row)

metrics_df = pd.DataFrame(metrics)

display(
    metrics_df.round(3)
)

,Model,MAE,RMSE,R2,MAPE_pct
0,Mean baseline,4000405.833,6028313.606,-0.337,4.309
1,OLS regression,3260295.901,4207898.338,0.348,3.655
2,Ridge regression,891421.351,1121793.014,0.954,1.024


In [18]:
predictions_df = pd.DataFrame({
    "month": test_df[DATE_COLUMN].dt.strftime("%Y-%m"),
    "actual_sales_lkr": y_test.to_numpy(),
    "baseline_sales_lkr": baseline_pred,
    "ols_predicted_sales_lkr": ols_pred,
    "ridge_predicted_sales_lkr": ridge_pred,
})

predictions_df["ridge_residual_lkr"] = (
    predictions_df["actual_sales_lkr"]
    - predictions_df["ridge_predicted_sales_lkr"]
)

display(predictions_df.round(2))

,month,actual_sales_lkr,baseline_sales_lkr,ols_predicted_sales_lkr,ridge_predicted_sales_lkr,ridge_residual_lkr
30,2025-07,82290486.0,84587359.37,8.309584e+07,82343418.84,-52932.84
31,2025-08,89134503.0,84587359.37,8.936524e+07,88321251.64,813251.36
32,2025-09,87603596.0,84587359.37,8.977560e+07,86055271.32,1548324.68
33,2025-10,84280236.0,84587359.37,8.696141e+07,86297244.35,-2017008.35
34,2025-11,84272664.0,84587359.37,9.082463e+07,84719961.00,-447297.00
35,2025-12,98107722.0,84587359.37,1.052283e+08,97638008.13,469713.87


In [19]:
predictions_df = pd.DataFrame({
    "month": test_df[DATE_COLUMN].dt.strftime("%Y-%m"),
    "actual_sales_lkr": y_test.to_numpy(),
    "baseline_sales_lkr": baseline_pred,
    "ols_predicted_sales_lkr": ols_pred,
    "ridge_predicted_sales_lkr": ridge_pred,
})

predictions_df["ridge_residual_lkr"] = (
    predictions_df["actual_sales_lkr"]
    - predictions_df["ridge_predicted_sales_lkr"]
)

display(predictions_df.round(2))

,month,actual_sales_lkr,baseline_sales_lkr,ols_predicted_sales_lkr,ridge_predicted_sales_lkr,ridge_residual_lkr
30,2025-07,82290486.0,84587359.37,8.309584e+07,82343418.84,-52932.84
31,2025-08,89134503.0,84587359.37,8.936524e+07,88321251.64,813251.36
32,2025-09,87603596.0,84587359.37,8.977560e+07,86055271.32,1548324.68
33,2025-10,84280236.0,84587359.37,8.696141e+07,86297244.35,-2017008.35
34,2025-11,84272664.0,84587359.37,9.082463e+07,84719961.00,-447297.00
35,2025-12,98107722.0,84587359.37,1.052283e+08,97638008.13,469713.87


In [21]:
coefficient_df = pd.DataFrame({
    "feature": FEATURES,
    "standardized_coefficient": (
        best_model.named_steps["regression"].coef_
    )
})

coefficient_df["feature_type"] = coefficient_df["feature"].apply(
    lambda name: (
        "Media"
        if name in MEDIA_FEATURES
        else "External factor"
    )
)

coefficient_df = coefficient_df.sort_values(
    "standardized_coefficient",
    key=lambda values: values.abs(),
    ascending=False
)

display(coefficient_df.round(4))

,feature,standardized_coefficient,feature_type
7,promotion_discount_pct,2.659638e+06,External factor
12,seasonal_demand_index,2.275469e+06,External factor
2,google_search_spend_lkr_hill,1.460642e+06,Media
0,tv_spend_lkr_hill,1.104297e+06,Media
3,youtube_spend_lkr_hill,1.023262e+06,Media
9,economic_sentiment_index_0_100,9.785545e+05,External factor
8,competitor_pressure_index_0_100,-8.050376e+05,External factor
4,radio_spend_lkr_hill,-7.926427e+05,Media
5,outdoor_spend_lkr_hill,6.998271e+05,Media
11,inflation_proxy_index,4.793130e+05,External factor


In [22]:
MODEL_DIR = ROOT / "outputs" / "models"
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# Save trained Ridge model
joblib.dump(
    best_model,
    MODEL_DIR / "ridge_mmm_baseline.joblib"
)

# Save transformation parameters
with open(
    MODEL_DIR / "transformation_params.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(TRANSFORM_PARAMS, file, indent=4)

# Save model configuration and feature list
model_config = {
    "target": TARGET,
    "date_column": DATE_COLUMN,
    "features": FEATURES,
    "media_features": MEDIA_FEATURES,
    "external_factors": EXTERNAL_FACTORS,
    "model_type": "Ridge Regression",
    "best_alpha": float(
        ridge_search.best_params_["regression__alpha"]
    ),
    "test_size_months": int(TEST_SIZE),
    "training_start": str(
        train_df[DATE_COLUMN].min().date()
    ),
    "training_end": str(
        train_df[DATE_COLUMN].max().date()
    ),
    "test_start": str(
        test_df[DATE_COLUMN].min().date()
    ),
    "test_end": str(
        test_df[DATE_COLUMN].max().date()
    )
}

with open(
    MODEL_DIR / "model_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(model_config, file, indent=4)

# Save evaluation results
metrics_df.to_csv(
    TABLE_DIR / "model_metrics.csv",
    index=False
)

predictions_df.to_csv(
    TABLE_DIR / "model_predictions.csv",
    index=False
)

coefficient_df.to_csv(
    TABLE_DIR / "model_coefficients.csv",
    index=False
)

# Save transformed dataset for audit/reuse
transformed_output = X.copy()
transformed_output.insert(
    0,
    DATE_COLUMN,
    df[DATE_COLUMN].values
)
transformed_output[TARGET] = df[TARGET].values

transformed_output.to_csv(
    TABLE_DIR / "model_features.csv",
    index=False
)

print("All modelling outputs saved successfully.")
print("Model folder:", MODEL_DIR)
print("Tables folder:", TABLE_DIR)

All modelling outputs saved successfully.
Model folder: d:\Gamage\marketing mix modeling\outputs\models
Tables folder: d:\Gamage\marketing mix modeling\outputs\tables
